# Comparação GraphRAG vs RAG ingênuo (benchmark)

Este notebook implementa **manualmente** a avaliação (sem RAGAS, LangSmith Evals, etc.):

**Recuperação (top-5)**
- **Recall@5**: entre os 5 primeiros `TextUnit` recuperados, quantos dos relevantes (gabarito) foram encontrados, dividido pelo número de trechos relevantes.
- **Precision@5**: entre as 5 posições, quantas são relevantes, dividido por 5.

**Geração**
- **F1 por tokens**: sobreposição de tokens (com contagem por frequência) entre resposta gerada e gabarito.

**Sistemas**
- **GraphRAG**: grafo LangGraph em `graphrag-neo4j-langchain` (`get_compiled_graph`).
- **RAG ingênuo**: busca vetorial direta nos nós `TextUnit` no Neo4j + mesmo prompt de síntese do projeto.

**Pré-requisitos**
- Indexar os `.txt` de `pdfs_txt/` (mesmo caminho absoluto definido em `DOCS_DIR` abaixo): `python scripts/run_indexing.py -i <pasta_docs>` dentro de `graphrag-neo4j-langchain`.
- Ficheiro `.env` em `graphrag-neo4j-langchain` com `NEO4J_*` e `OPENAI_API_KEY`.
- Perguntas/gabaritos: `docs/jgab_benchmark_qa.csv` (ou o CSV que você estiver usando no benchmark atual).

In [9]:
from __future__ import annotations

import csv
import re
import sys
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

# Raiz do repositório benchmarking-graphrag (kernel: cwd = raiz ou pasta notebooks/)
_here = Path.cwd()
if (_here / "graphrag-neo4j-langchain" / "src").is_dir():
    PROJECT_ROOT = _here.resolve()
elif (_here.parent / "graphrag-neo4j-langchain" / "src").is_dir():
    PROJECT_ROOT = _here.parent.resolve()
else:
    raise FileNotFoundError(
        "Defina manualmente PROJECT_ROOT para a pasta que contém graphrag-neo4j-langchain/ "
        f"(cwd atual: {_here})"
    )

GRAPHRAG_SRC = PROJECT_ROOT / "graphrag-neo4j-langchain" / "src"
if not GRAPHRAG_SRC.is_dir():
    raise FileNotFoundError(f"Não encontrei graphrag em: {GRAPHRAG_SRC}")

if str(GRAPHRAG_SRC) not in sys.path:
    sys.path.insert(0, str(GRAPHRAG_SRC))

_env = PROJECT_ROOT / "graphrag-neo4j-langchain" / ".env"
if _env.exists():
    try:
        from dotenv import load_dotenv

        load_dotenv(_env)
    except ImportError:
        pass

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\ADM\Documents\benchmarking-graphrag


## 1. Métricas (implementação direta)

In [10]:
def normalize_tokens(text: str) -> list[str]:
    t = (text or "").lower()
    t = re.sub(r"[^\w\s\u00C0-\u024F]", " ", t, flags=re.UNICODE)
    return [x for x in t.split() if x]


def token_f1(reference: str, predicted: str) -> float:
    """F1 por tokens (multiset)."""
    ref = normalize_tokens(reference)
    hyp = normalize_tokens(predicted)
    if not ref and not hyp:
        return 1.0
    if not ref or not hyp:
        return 0.0
    cr, ch = Counter(ref), Counter(hyp)
    overlap = sum((cr & ch).values())
    prec = overlap / max(len(hyp), 1)
    rec = overlap / max(len(ref), 1)
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)


def recall_at_k(retrieved_ids: list[str], gold_ids: set[str], k: int) -> float:
    top = retrieved_ids[:k]
    hits = sum(1 for x in top if x in gold_ids)
    if not gold_ids:
        return 1.0
    return hits / len(gold_ids)


def precision_at_k(retrieved_ids: list[str], gold_ids: set[str], k: int) -> float:
    top = retrieved_ids[:k]
    if k <= 0:
        return 0.0
    hits = sum(1 for x in top if x in gold_ids)
    return hits / k

## 2. Dados de avaliação e mapeamento chunk → `TextUnit.id`

In [11]:
@dataclass
class QARow:
    question: str
    reference_answer: str
    hop_type: str
    gold_chunk_indices: list[int]


def load_qa_csv(path: Path) -> list[QARow]:
    rows: list[QARow] = []
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        req = {"question", "reference_answer", "hop_type", "relevant_chunk_indices"}
        miss = req - set(reader.fieldnames or [])
        if miss:
            raise ValueError(f"CSV incompleto. Faltam: {sorted(miss)}")
        for r in reader:
            q = (r.get("question") or "").strip()
            ra = (r.get("reference_answer") or "").strip()
            hop = (r.get("hop_type") or "").strip()
            idx_raw = (r.get("relevant_chunk_indices") or "").strip()
            if not q or not ra:
                continue
            indices = [int(x.strip()) for x in idx_raw.split(",") if x.strip().isdigit()]
            rows.append(QARow(question=q, reference_answer=ra, hop_type=hop, gold_chunk_indices=indices))
    if not rows:
        raise ValueError("Nenhuma linha válida no CSV.")
    return rows


def build_chunk_index_to_tu_id(doc_path: Path) -> dict[int, str]:
    """Ids alinhados ao pipeline: {caminho_absoluto_txt}_{índice}."""
    from langchain_core.documents import Document
    from graphrag.indexing.load_and_chunk import chunk_documents

    doc_path = doc_path.resolve()
    text = doc_path.read_text(encoding="utf-8")
    doc = Document(page_content=text, metadata={"source": str(doc_path)})
    chunks = chunk_documents([doc])
    base = str(doc_path)
    return {i: f"{base}_{i}" for i in range(len(chunks))}

## 3. Recuperação e geração: RAG ingênuo vs GraphRAG

In [12]:
def naive_retrieve_top_k(question: str, k: int) -> tuple[list[str], list[str]]:
    from graphrag.store.vector_index import get_vector_index_text_units

    store = get_vector_index_text_units()
    if store is None:
        raise RuntimeError("Índice vetorial TextUnit indisponível. Execute embed no pipeline.")
    docs = store.similarity_search(question.strip(), k=k)
    ids, texts = [], []
    for d in docs:
        md = getattr(d, "metadata", None) or {}
        tid = md.get("id") or md.get("t.id")
        ids.append(str(tid) if tid is not None else "")
        texts.append(getattr(d, "page_content", "") or "")
    return ids, texts


def graphrag_top_k_text_unit_ids(state: dict, k: int) -> list[str]:
    out: list[str] = []
    for doc in state.get("context_docs") or []:
        if not isinstance(doc, dict):
            continue
        md = doc.get("metadata") or {}
        if not isinstance(md, dict):
            continue
        kind = str(md.get("kind") or "")
        if not kind.startswith("text_unit"):
            continue
        tid = md.get("text_unit_id")
        if not tid:
            continue
        s = str(tid)
        if s not in out:
            out.append(s)
        if len(out) >= k:
            break
    return out[:k]


def run_graphrag(question: str, thread_id: str) -> dict:
    from graphrag.graph.query_graph import get_compiled_graph

    compiled = get_compiled_graph()
    cfg = {"configurable": {"thread_id": f"nb-{thread_id}"}}
    return compiled.invoke({"question": question}, config=cfg)


def naive_generate(question: str, context_chunks: list[str]) -> str:
    from graphrag.config import OPENAI_API_KEY, LLM_MODEL
    from graphrag.monitoring.token_cost import tracked_chat_openai
    from graphrag.prompts.synthesis import SYNTHESIS_PROMPT

    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY não definida.")
    ctx = "\n\n".join(context_chunks) if context_chunks else "(sem contexto)"
    llm = tracked_chat_openai(model=LLM_MODEL, temperature=0, api_key=OPENAI_API_KEY)
    chain = SYNTHESIS_PROMPT | llm
    msg = chain.invoke({"context": ctx, "question": question})
    return getattr(msg, "content", str(msg)) or ""

## 4. Parâmetros e validação do ambiente

In [13]:
K = 5
DOCS_DIR = (PROJECT_ROOT / "pdfs_txt").resolve()
DOC_FILENAME: str | None = None  # Ex.: "phmsa_20260075.txt". None = usa o primeiro .txt em ordem alfabética.
QA_CSV = (PROJECT_ROOT / "docs" / "jgab_benchmark_qa.csv").resolve()

assert DOCS_DIR.is_dir(), f"Falta a pasta de documentos: {DOCS_DIR}"
assert QA_CSV.is_file(), f"Falta o CSV: {QA_CSV}"

from graphrag.config import OPENAI_API_KEY

assert OPENAI_API_KEY, "Configure OPENAI_API_KEY no .env do graphrag-neo4j-langchain."

txt_files = sorted(DOCS_DIR.glob("*.txt"))
assert txt_files, f"Nenhum .txt encontrado em: {DOCS_DIR}"

if DOC_FILENAME:
    DOC_PATH = (DOCS_DIR / DOC_FILENAME).resolve()
    assert DOC_PATH.is_file(), f"Arquivo selecionado não encontrado: {DOC_PATH}"
else:
    DOC_PATH = txt_files[0].resolve()

idx_to_tu = build_chunk_index_to_tu_id(DOC_PATH)
qa_rows = load_qa_csv(QA_CSV)

print(f"Pasta de entrada: {DOCS_DIR}")
print(f"Documento avaliado: {DOC_PATH.name}")
print(f"Arquivos .txt disponíveis: {len(txt_files)}")
print(f"Chunks locais: {len(idx_to_tu)} | Perguntas: {len(qa_rows)}")

AssertionError: Configure OPENAI_API_KEY no .env do graphrag-neo4j-langchain.

## 5. Correr a comparação (loop explícito)

In [ ]:
results: list[dict] = []

for i, row in enumerate(qa_rows):
    gidx = sorted(set(row.gold_chunk_indices))
    missing = set(gidx) - set(idx_to_tu.keys())
    if missing:
        raise ValueError(f"Q{i+1}: índices inválidos {sorted(missing)}")
    gold_ids = {idx_to_tu[j] for j in gidx}
    tag = str(abs(hash(row.question)) % 10_000_000)

    n_ids, n_texts = naive_retrieve_top_k(row.question, K)
    n_rec = recall_at_k(n_ids, gold_ids, K)
    n_prec = precision_at_k(n_ids, gold_ids, K)
    n_ans = naive_generate(row.question, n_texts)
    n_f1 = token_f1(row.reference_answer, n_ans)

    g_state = run_graphrag(row.question, tag)
    g_ids = graphrag_top_k_text_unit_ids(g_state, K)
    g_ans = (g_state.get("final_answer") or "").strip()
    g_rec = recall_at_k(g_ids, gold_ids, K)
    g_prec = precision_at_k(g_ids, gold_ids, K)
    g_f1 = token_f1(row.reference_answer, g_ans)

    results.append(
        {
            "q": i + 1,
            "hop": row.hop_type,
            "system": "naive_rag",
            "recall@5": n_rec,
            "precision@5": n_prec,
            "token_f1": n_f1,
            "question": row.question,
            "reference": row.reference_answer,
            "generated": n_ans,
            "retrieved_ids": " | ".join(n_ids),
        }
    )
    results.append(
        {
            "q": i + 1,
            "hop": row.hop_type,
            "system": "graphrag",
            "recall@5": g_rec,
            "precision@5": g_prec,
            "token_f1": g_f1,
            "question": row.question,
            "reference": row.reference_answer,
            "generated": g_ans,
            "retrieved_ids": " | ".join(g_ids),
        }
    )
    print(f"[{i+1}/{len(qa_rows)}] {row.hop_type}  naive R@5={n_rec:.3f}  graphrag R@5={g_rec:.3f}")

print("Concluído.")

## 6. Agregar e visualizar (tabela simples)

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None

if pd is not None:
    df = pd.DataFrame(results)
    display_cols = ["q", "hop", "system", "recall@5", "precision@5", "token_f1"]
    display(df[display_cols])

    summary = (
        df.groupby("system")[["recall@5", "precision@5", "token_f1"]]
        .mean()
        .rename_axis("média macro")
    )
    display(summary)
else:
    from statistics import mean

    for sys in ("naive_rag", "graphrag"):
        sub = [r for r in results if r["system"] == sys]
        print(
            sys,
            "recall@5", mean(x["recall@5"] for x in sub),
            "precision@5", mean(x["precision@5"] for x in sub),
            "token_f1", mean(x["token_f1"] for x in sub),
        )